### Looking into character ids and similarity with token ids 

In [3]:
with open("data/sherlock.txt", "r", encoding="utf-8") as f:
    text = f.read()

byte_test = bytearray(text, "utf-8")
ids = list(byte_test)
print(text[100:110])
print(ids[100:110])

 Conan Doy
[32, 67, 111, 110, 97, 110, 32, 68, 111, 121]


### Using tiktoken to view how tokens of gpt2 and their respective ids look

In [4]:
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300, 310):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")

300:  l
301: st
302:  re
303: ve
304:  e
305: ro
306: ly
307:  be
308:  g
309:  T


### Creating custom and basic preprocessing unit starting here

In [5]:
import re

print(text[:262], "\n")

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
preprocessed = [i.strip() for i in preprocessed if i.strip()]
print(preprocessed[:20])





                        THE ADVENTURES OF SHERLOCK HOLMES

                               Arthur Conan Doyle



                                Table of contents

               A Scandal in Bohemia
               The Red-Headed League
               A Case  

['THE', 'ADVENTURES', 'OF', 'SHERLOCK', 'HOLMES', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'A', 'Scandal', 'in', 'Bohemia', 'The', 'Red-Headed', 'League', 'A', 'Case']


### From the tokens we created, we need to get only unique ones and sort them

In [6]:
sorted_ = sorted(set(preprocessed))
print(len(sorted_))

vocab = {v:k for k, v in enumerate(sorted_)}
print(vocab.items())

8804
dict_items([('!', 0), ('"', 1), ('&', 2), ("'", 3), ('(', 4), (')', 5), (',', 6), ('--', 7), ('-the-wisp', 8), ('.', 9), ('//sherlock-holm', 10), ('000', 11), ('1', 12), ('10', 13), ('1000', 14), ('10s', 15), ('10th', 16), ('11', 17), ('117', 18), ('12s', 19), ('12th', 20), ('14', 21), ('140', 22), ('15', 23), ('150', 24), ('16A', 25), ('17', 26), ('1846', 27), ('1858', 28), ('1869', 29), ('1870', 30), ('1878', 31), ('1883', 32), ('1884', 33), ('1887', 34), ('1888', 35), ('1890', 36), ('19th', 37), ('1s', 38), ('2', 39), ('221b', 40), ('226', 41), ('22nd', 42), ('249', 43), ('26', 44), ('26s', 45), ('27', 46), ('270', 47), ('2nd', 48), ('2s', 49), ('3', 50), ('30', 51), ('31', 52), ('35', 53), ('3rd', 54), ('4', 55), ('421', 56), ('4d', 57), ('4th', 58), ('4½', 59), ('5', 60), ('6', 61), ('60', 62), ('6d', 63), ('7', 64), ('77', 65), ('7s', 66), ('7th', 67), ('82', 68), ('83', 69), ('84', 70), ('85', 71), ('87', 72), ('89', 73), ('8d', 74), ('8s', 75), ('9', 76), ('90', 77), ('9th

### Putting them all together into a simple tokenizer class

In [7]:
class BaseTokenizerV1:
    def __init__(self, vocab):
        self.token_to_id = vocab
        self.id_to_token = {v:k for k, v in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [i.strip() for i in preprocessed if i.strip()]
        ids = [self.token_to_id[t] for t in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.id_to_token[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

### Since we are considering the text data as the corpos and assuming those are the available vocab, we need to handle tokens that are not in the vocab, when we try to encode or decode

In [8]:
sorted_.extend(["<|EOT|>", "<|UNKNOWN|>"])
vocab = {v:k for k, v in enumerate(sorted_)}

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('£700', 8801)
('£750', 8802)
('£88', 8803)
('<|EOT|>', 8804)
('<|UNKNOWN|>', 8805)


### So lets redo the tokenizer with this new change and test

In [9]:
class BaseTokenizerV2:
    def __init__(self, vocab):
        self.token_to_id = vocab
        self.id_to_token = {v:k for k, v in vocab.items()}

    def encode(self, text):
        text = re.sub(r'\s+', ' ', text).strip()
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [i.strip() for i in preprocessed if i.strip()]
        preprocessed = [t if t in self.token_to_id else "<|UNKNOWN|>" for t in preprocessed]
        ids = [self.token_to_id[t] for t in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.id_to_token[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [10]:
tokenizer = BaseTokenizerV2(vocab)

text1 = "This is a test sentence."
text2 = "I wanted to share my name with you, I am sagar!"

text = " <|EOT|> ".join([text1, text2])
print(text)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

This is a test sentence. <|EOT|> I wanted to share my name with you, I am sagar!
[1089, 4835, 1230, 7868, 6991, 9, 8804, 593, 8466, 7993, 7053, 5519, 5529, 8673, 8770, 6, 593, 1441, 8805, 0]
This is a test sentence. <|EOT|> I wanted to share my name with you, I am <|UNKNOWN|>!


### Since we have created our custom tokenizer, we won't be using that in production so let's move on to tiktoken which uses BPE

### As <|endoftext|> this is a special token in gpt2 tokenizer, it can be specified as special token else it will split that token

In [11]:
tokenizer = tiktoken.get_encoding("gpt2")

text1 = "This"
text2 = "Sagar"

text = " <|endoftext|> ".join([text1, text2])

ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(ids)
print(tokenizer.decode(ids))

[1212, 220, 50256, 25605, 283]
This <|endoftext|> Sagar


### Data sampling

In [12]:
with open("data/sherlock.txt", "r", encoding="utf-8") as f:
    text = f.read()

encoded_text = tokenizer.encode(text)
print(len(encoded_text))

179388


### Lets say we have context_size as 4, how shall we sample inputs and target from the series of encoded tokens

In [13]:
context_size = 4

x = encoded_text[:context_size]
y = encoded_text[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}", "\n")

for i in range(1, context_size):
    input_ids = x[:i]
    target_ids = y[i]
    print(input_ids, "--->", target_ids)

x: [628, 628, 220, 220]
y: [628, 220, 220, 220] 

[628] ---> 220
[628, 628] ---> 220
[628, 628, 220] ---> 220


### Dataset and data loading

In [14]:
from torch.utils.data import Dataset
import torch

class BaseDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_size, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > context_size, "Number of tokenized inputs must at least be equal to max_length+1"

        for i in range(1, len(token_ids)-context_size, stride):
            x = token_ids[i:i+context_size]
            y = token_ids[i+1:i+context_size+1]
            self.input_ids.append(torch.tensor(x))
            self.target_ids.append(torch.tensor(y))
        
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, i):
        return self.input_ids[i], self.target_ids[i]

In [15]:
from torch.utils.data import DataLoader

def createDataLoaderV1(text, batch_size=4, context_size=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = BaseDatasetV1(text, tokenizer, context_size, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    
    return dataloader


### So lets test our data loader with stride as 1

In [ ]:
with open("data/sherlock.txt", "r", encoding="utf-8") as f:
    text = f.read()

dataloader = createDataLoaderV1(text, batch_size=1, context_size=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

dataloader = createDataLoaderV1(text, batch_size=1, context_size=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

for _ in range(100):
    _batch = next(data_iter)
print(_batch)
for _ in range(100):
    _batch = next(data_iter)
print(_batch)

[tensor([[628, 220, 220, 220]]), tensor([[220, 220, 220, 220]])]
[tensor([[ 220,  220, 8655,  286]]), tensor([[  220,  8655,   286, 10154]])]


### dataloader with stride 8

In [17]:
dataloader = createDataLoaderV1(text, batch_size=8, context_size=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  628,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [ 3336, 43685,  3525, 29514],
        [ 3963,  6006,  1137, 36840]])

Targets:
 tensor([[  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,  3336],
        [43685,  3525, 29514,  3963],
        [ 6006,  1137, 36840, 49707]])


### Okay we are done with data loading and now moving on to the next step - token embeddings

In [18]:
with open("data/sherlock.txt", "r", encoding="utf-8") as f:
    text = f.read()

vocab_size = 50257
output_dim = 256
batch_size = 8
context_size = 16

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_size, output_dim)

dataloader = createDataLoaderV1(
    text,
    batch_size=batch_size,
    context_size=context_size,
    stride=context_size,
    shuffle=False
)

In [21]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(context_size))

    input_embeddings = token_embeddings + pos_embeddings

    break

print(input_embeddings.shape)
print(pos_embeddings.shape)
print(token_embeddings.shape)

torch.Size([8, 16, 256])
torch.Size([16, 256])
torch.Size([8, 16, 256])


### This covers the basics for preprocessing